In [17]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [18]:
NUM_OF_TOKENS = 120

In [19]:
def get_token_df(file_path, body_bins=11, num_bins=5, return_edges=False, input_edges=False):
    df = pd.read_csv(file_path)
    df['body'] = df['close_normalized'] - df['open_normalized']
    df['wick'] = df['high_normalized'] - df[['open_normalized', 'close_normalized']].max(axis=1)
    df['shadow'] = df[['open_normalized', 'close_normalized']].min(axis=1) - df['low_normalized']
    df.drop(['open_normalized', 'high_normalized', 'low_normalized', 'close_normalized'], inplace=True, axis=1)

    if input_edges:
        body_bin_edges = input_edges['body_bin_edges']
        wick_bin_edges = input_edges['wick_bin_edges']
        shadow_bin_edges = input_edges['shadow_bin_edges']
    else:
        _, body_bin_edges = pd.qcut(df['body'], body_bins, retbins=True, duplicates='drop')
        __, wick_bin_edges = pd.qcut(df['wick'], num_bins, retbins=True, duplicates='drop')
        ___, shadow_bin_edges = pd.qcut(df['shadow'], num_bins, retbins=True, duplicates='drop')

    df['wick_bins'] = pd.cut(df['wick'], bins=wick_bin_edges, labels=False, include_lowest=True)
    df['body_bins'] = pd.cut(df['body'], bins=body_bin_edges, labels=False, include_lowest=True)
    df['shadow_bins'] = pd.cut(df['shadow'], bins=shadow_bin_edges, labels=False, include_lowest=True)

    # Token as string
    df['token'] = df[['body_bins', 'wick_bins', 'shadow_bins']].astype(str).agg('_'.join, axis=1)
    
    # Map tokens to unique integers
    token_mapping = {token: idx for idx, token in enumerate(df['token'].unique())}
    df['token_id'] = df['token'].map(token_mapping)
    
    df.drop(['wick', 'shadow', 'body', 'wick_bins', 'shadow_bins', 'body_bins', 'token'], inplace=True, axis=1)
    
    if return_edges:
        return {"body_bin_edges": body_bin_edges, "shadow_bin_edges": shadow_bin_edges, "wick_bin_edges": wick_bin_edges}, df, token_mapping
    else:
        return df, token_mapping


In [20]:
from imblearn.over_sampling import RandomOverSampler

def balance_with_oversampler(df, target_col):
    oversampler = RandomOverSampler()
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Perform oversampling
    X_resampled, y_resampled = oversampler.fit_resample(X, y)
    
    # Recreate a balanced DataFrame
    df_balanced = pd.concat([pd.DataFrame(X_resampled), pd.DataFrame(y_resampled, columns=[target_col])], axis=1)
    
    return df_balanced

In [21]:
def get_ready_data(df, num_prev=NUM_OF_TOKENS-1, shuffle=False, over_sample=False):
    for i in range(1, num_prev + 1):
        df[f'token-{i}'] = df['token_id'].shift(i)
        
    df.dropna(inplace=True)
    if over_sample:
        df = balance_with_oversampler(df, 'target')

    if shuffle:
        df = df.sample(frac=1).reset_index(drop=True)

    # Return the target and the shifted token columns
    return df['target'], df.drop(columns=['target'])


In [22]:
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Dropout, Embedding
from tensorflow.keras.models import Model
import numpy as np

# Positional Encoding remains the same
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, maxlen, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(maxlen, d_model)

    def positional_encoding(self, maxlen, d_model):
        import numpy as np
        positions = np.arange(maxlen)[:, np.newaxis]
        angles = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (angles // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

# Transformer Block remains the same
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Transformer Model
def build_transformer_model(input_shape, num_classes, vocab_size, d_model=32, num_heads=2, ff_dim=64, num_layers=2):
    inputs = Input(shape=input_shape)  # Input is a sequence of token IDs
    
    # Embedding layer to map token IDs to dense vectors
    x = Embedding(input_dim=vocab_size, output_dim=d_model)(inputs)
    
    # Positional Encoding
    x = PositionalEncoding(input_shape[0], d_model)(x)
    
    # Transformer blocks
    for _ in range(num_layers):
        x = TransformerBlock(d_model, num_heads, ff_dim)(x)
    
    x = Dense(128, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.1)(x)
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model



In [23]:
# Load and process the data
edges, train_df, train_mapping = get_token_df('training/ADAUSD/train.csv', return_edges=True)
val_df, _ = get_token_df('training/ADAUSD/val.csv', input_edges=edges)
test_df, _ = get_token_df('training/ADAUSD/test.csv', input_edges=edges)

# Prepare data for training
train_targets, train_features = get_ready_data(train_df, shuffle=True)
val_targets, val_features = get_ready_data(val_df)
test_targets, test_features = get_ready_data(test_df)

# Convert the features to NumPy arrays for training
train_features = train_features.to_numpy()
val_features = val_features.to_numpy()
test_features = test_features.to_numpy()


/tmp/ipykernel_348799/1791793380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'token-{i}'] = df['token_id'].shift(i)
/tmp/ipykernel_348799/1791793380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'token-{i}'] = df['token_id'].shift(i)
/tmp/ipykernel_348799/1791793380.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented

In [24]:
# Define the model input shape and vocab size
input_shape = (NUM_OF_TOKENS,)  # The number of tokens used in the sequence
vocab_size = len(train_mapping)  # Number of unique tokens

# Build and compile the model
model = build_transformer_model(input_shape, num_classes=3, vocab_size=vocab_size, d_model=128, ff_dim=256, num_layers=8, num_heads=4)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

class_weights = {
    0: 1,
    1: 2,
    2: 2
}

# Train the model
history = model.fit(
    train_features, train_targets,
    validation_data=(val_features, val_targets),
    epochs=10,
    batch_size=64,
    class_weight = class_weights
)


Epoch 1/10
14313/14313 [==============================] - 2456s 170ms/step - loss: 0.9497 - accuracy: 0.8267 - val_loss: 0.5314 - val_accuracy: 0.8809
Epoch 2/10
14313/14313 [==============================] - 23599s 2s/step - loss: 0.9471 - accuracy: 0.8268 - val_loss: 0.5332 - val_accuracy: 0.8809
Epoch 3/10
 7585/14313 [==============>...............] - ETA: 16:21 - loss: 0.9453 - accuracy: 0.8273

KeyboardInterrupt: 